In [1]:
df_orders_raw = spark.read.csv(
    "Files/Bronze/orders_data.csv",
    header=True,
    inferSchema=True
)
display(df_orders_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a7726764-3f32-464e-885c-55277dcd7bb2)

In [2]:
df_inventory_raw = spark.read.parquet(
    "Files/Bronze/inventory_data.parquet"
)
display(df_inventory_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5bc4bf81-aa4d-4b5f-b3d9-2425bae434a6)

In [3]:
df_return_raw = spark.read.parquet(
    "Files/Bronze/returns_data.xlsx.parquet"
)
display(df_return_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 409fc903-b101-4620-a951-b3d8a0b0a1f2)

In [4]:
df_inventory_raw.printSchema()

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 6, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- productName: string (nullable = true)
 |-- stock: string (nullable = true)
 |-- last_stocked: string (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- available: string (nullable = true)



In [5]:

display(df_inventory_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c554df7b-fc3c-4bfa-8a5e-1282f713f030)

In [6]:
df_inventory_raw.printSchema()

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 8, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- productName: string (nullable = true)
 |-- stock: string (nullable = true)
 |-- last_stocked: string (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- available: string (nullable = true)



In [8]:
df_inventory_raw.printSchema()

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 10, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- productName: string (nullable = true)
 |-- stock: string (nullable = true)
 |-- last_stocked: string (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- available: string (nullable = true)



In [9]:
display(df_inventory_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff851487-4885-4805-9d81-ee587543762a)

In [10]:
df_inventory = spark.table("silver_inventory")
display(df_inventory)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 17f33102-aaae-4355-96b7-28d6b1811b28)

In [11]:
df_inventory_raw.printSchema()

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 13, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- productName: string (nullable = true)
 |-- stock: string (nullable = true)
 |-- last_stocked: string (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- available: string (nullable = true)



In [12]:
from pyspark.sql.functions import *

df_inventory = (
    df_inventory_raw

    # Standardize column names
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("last_stocked", "LastStocked")

    # Clean Stock
    .withColumn(
        "Stock",
        when(
            col("stock").rlike("^[0-9]+$"),
            col("stock").cast("integer")
        )
        .when(
            col("stock").isNull() | (trim(col("stock")) == ""),
            lit(None)
        )
        .otherwise(
            when(lower(col("stock")).contains("twenty five"), 25)
            .when(lower(col("stock")).contains("twenty"), 20)
            .when(lower(col("stock")).contains("eighteen"), 18)
            .when(lower(col("stock")).contains("fifteen"), 15)
            .when(lower(col("stock")).contains("twelve"), 12)
            .otherwise(None)
        )
    )

    # Clean Last Stocked date
    .withColumn(
        "LastStocked",
        to_date(
            regexp_replace(col("LastStocked"), "[./]", "-"),
            "yyyy-MM-dd"
        )
    )

    # Clean Cost Price
    .withColumn(
        "CostPrice",
        regexp_extract(
            col("CostPrice"),
            r"(\d+\.?\d*)",
            1
        ).cast("double")
    )

    # Clean Warehouse
    .withColumn(
        "Warehouse",
        initcap(
            trim(
                regexp_replace(
                    col("warehouse"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )

    # Convert Available to Boolean
    .withColumn(
        "Available",
        when(
            lower(trim(col("available"))).isin(
                "yes", "y", "true"
            ),
            True
        )
        .when(
            lower(trim(col("available"))).isin(
                "no", "n", "false"
            ),
            False
        )
        .otherwise(None)
    )

    # Remove raw columns
    .drop("stock", "warehouse", "available")
)

display(df_inventory)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 818e71ac-2875-45f1-b73e-1c8f437c2532)

In [14]:
df_inventory.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_inventory")

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 16, Finished, Available, Finished, False)

In [15]:
spark.sql("DESCRIBE silver_inventory").show(truncate=False)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 17, Finished, Available, Finished, False)

+-----------+---------+-------+
|col_name   |data_type|comment|
+-----------+---------+-------+
|product_id |string   |NULL   |
|ProductName|string   |NULL   |
|LastStocked|date     |NULL   |
|CostPrice  |double   |NULL   |
+-----------+---------+-------+



In [16]:
df_return_raw.printSchema()

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 18, Finished, Available, Finished, False)

root
 |-- Prop_0: string (nullable = true)
 |-- Prop_1: string (nullable = true)
 |-- Prop_2: string (nullable = true)
 |-- Prop_3: string (nullable = true)
 |-- Prop_4: string (nullable = true)
 |-- Prop_5: string (nullable = true)
 |-- Prop_6: string (nullable = true)
 |-- Prop_7: string (nullable = true)
 |-- Prop_8: string (nullable = true)



In [17]:
display(df_return_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1b60504f-7045-4277-ae3d-2b9d6bb1059e)

In [18]:
display(df_return_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d173cad6-a8b6-4b75-9b75-67579beb2ae5)

In [20]:
from pyspark.sql.functions import *

df_returns = (
    df_return_raw

    # Remove the first row because it contains the column headings
    .filter(col("Prop_0") != "Return_ID")

    # Rename Prop columns
    .withColumnRenamed("Prop_0", "ReturnID")
    .withColumnRenamed("Prop_1", "OrderID")
    .withColumnRenamed("Prop_2", "CustomerID")
    .withColumnRenamed("Prop_3", "Product")
    .withColumnRenamed("Prop_4", "ReturnReason")
    .withColumnRenamed("Prop_5", "ReturnDate")
    .withColumnRenamed("Prop_6", "RefundStatus")
    .withColumnRenamed("Prop_7", "PickupAddress")
    .withColumnRenamed("Prop_8", "ReturnAmount")

    # Clean IDs
    .withColumn("ReturnID", trim(col("ReturnID")))
    .withColumn("OrderID", trim(col("OrderID")))
    .withColumn("CustomerID", trim(upper(col("CustomerID"))))

    # Convert NULL text to actual null
    .withColumn(
        "OrderID",
        when(
            upper(trim(col("OrderID"))) == "NULL",
            None
        ).otherwise(col("OrderID"))
    )

    .withColumn(
        "CustomerID",
        when(
            upper(trim(col("CustomerID"))) == "NULL",
            None
        ).otherwise(col("CustomerID"))
    )

    # Clean Product
    .withColumn(
        "Product",
        initcap(trim(col("Product")))
    )

    # Clean Return Reason
    .withColumn(
        "ReturnReason",
        initcap(trim(col("ReturnReason")))
    )

    # Convert mixed date formats
    .withColumn(
        "ReturnDate",
        coalesce(
            to_date(col("ReturnDate"), "dd-MM-yyyy"),
            to_date(col("ReturnDate"), "dd.MM.yyyy"),
            to_date(col("ReturnDate"), "yyyy.MM.dd"),
            to_date(col("ReturnDate"), "yyyy-MM-dd"),
            to_date(col("ReturnDate"), "MM/dd/yyyy"),
            to_date(col("ReturnDate"), "MMMM dd yyyy"),
            to_date(col("ReturnDate"), "yyyy/MM/dd")
        )
    )

    # Clean Refund Status
    .withColumn(
        "RefundStatus",
        lower(
            trim(
                regexp_replace(
                    col("RefundStatus"),
                    r"[^a-zA-Z ]",
                    ""
                )
            )
        )
    )

    # Clean Pickup Address
    .withColumn(
        "PickupAddress",
        initcap(
            trim(
                regexp_replace(
                    col("PickupAddress"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )

    # Clean Return Amount
    .withColumn(
        "ReturnAmount",
        regexp_extract(
            col("ReturnAmount"),
            r"(\d+\.?\d*)",
            1
        ).cast("double")
    )

    # Remove invalid Return IDs
    .filter(col("ReturnID").isNotNull())
)

display(df_returns)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ef014290-e093-4749-b61e-e83acfe711d8)

In [21]:
df_returns.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_returns")

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 23, Finished, Available, Finished, False)

In [22]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 24, Finished, Available, Finished, False)

+-----------------------+----------------+-----------+
|namespace              |tableName       |isTemporary|
+-----------------------+----------------+-----------+
|retailWS.Retail_LH1.dbo|silver_inventory|false      |
|retailWS.Retail_LH1.dbo|silver_orders   |false      |
|retailWS.Retail_LH1.dbo|silver_returns  |false      |
+-----------------------+----------------+-----------+



In [25]:
max("Stock").alias("Current_Stock")

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 27, Finished, Available, Finished, False)

Column<'max(Stock) AS Current_Stock'>

In [26]:
df_inventory = spark.table("silver_inventory")

df_inventory.printSchema()

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 28, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- LastStocked: date (nullable = true)
 |-- CostPrice: double (nullable = true)



In [27]:
display(df_inventory)v

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48e449ac-0cbc-49a0-8660-1d37cb3d6794)

In [28]:
from pyspark.sql.functions import *

# Rebuild Inventory Silver table from the Bronze data
df_inventory = (
    df_inventory_raw
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("last_stocked", "LastStocked")

    # Clean Stock
    .withColumn(
        "Stock",
        when(
            col("stock").rlike("^[0-9]+$"),
            col("stock").cast("integer")
        )
        .when(
            col("stock").isNull() | (trim(col("stock")) == ""),
            lit(None).cast("integer")
        )
        .when(
            lower(col("stock")).contains("twenty five"),
            lit(25)
        )
        .when(
            lower(col("stock")).contains("twenty"),
            lit(20)
        )
        .when(
            lower(col("stock")).contains("eighteen"),
            lit(18)
        )
        .when(
            lower(col("stock")).contains("fifteen"),
            lit(15)
        )
        .when(
            lower(col("stock")).contains("twelve"),
            lit(12)
        )
        .otherwise(lit(None).cast("integer"))
    )

    # Clean date
    .withColumn(
        "LastStocked",
        coalesce(
            to_date(col("LastStocked"), "yyyy-MM-dd"),
            to_date(col("LastStocked"), "dd-MM-yyyy"),
            to_date(col("LastStocked"), "dd/MM/yyyy")
        )
    )

    # Clean CostPrice
    .withColumn(
        "CostPrice",
        regexp_extract(
            col("CostPrice"),
            r"(\d+\.?\d*)",
            1
        ).cast("double")
    )

    # Clean Warehouse
    .withColumn(
        "Warehouse",
        initcap(
            trim(
                regexp_replace(
                    col("warehouse"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )

    # Clean Available
    .withColumn(
        "Available",
        when(
            lower(trim(col("available"))).isin(
                "yes", "y", "true"
            ),
            True
        )
        .when(
            lower(trim(col("available"))).isin(
                "no", "n", "false"
            ),
            False
        )
        .otherwise(None)
    )

    # Remove original messy columns
    .drop("stock", "warehouse", "available")
)

display(df_inventory)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c31238f2-7ffc-4548-8c6d-796df6fd8199)

In [29]:
df_inventory.printSchema()

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 31, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- LastStocked: date (nullable = true)
 |-- CostPrice: double (nullable = true)



In [30]:
spark.sql("DROP TABLE IF EXISTS silver_inventory")

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 32, Finished, Available, Finished, False)

DataFrame[]

In [31]:
df_inventory.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_inventory")

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 33, Finished, Available, Finished, False)

In [32]:
spark.table("silver_inventory").printSchema()

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 34, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- LastStocked: date (nullable = true)
 |-- CostPrice: double (nullable = true)



In [34]:
from pyspark.sql.functions import *

df_orders = spark.table("silver_orders")
df_orders = df_orders.withColumnRenamed("Order_ID", "OrderID")

df_returns = spark.table("silver_returns")
df_inventory = spark.table("silver_inventory")

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 36, Finished, Available, Finished, False)

In [37]:
# Check the current inventory table
df_inventory_check = spark.table("silver_inventory")

print("Current columns:")
print(df_inventory_check.columns)

display(df_inventory_check)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 39, Finished, Available, Finished, False)

Current columns:
['product_id', 'ProductName', 'LastStocked', 'CostPrice']


SynapseWidget(Synapse.DataFrame, 88c4d52f-8f1c-4f4d-841a-275f25e0f53a)

In [39]:
from pyspark.sql.functions import col, when, lit, regexp_extract

# Read the ORIGINAL Bronze inventory file
df_inventory_raw = spark.read.parquet(
    "Files/Bronze/inventory_data.parquet"
)

# Check the original columns
print(df_inventory_raw.columns)

display(df_inventory_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 41, Finished, Available, Finished, False)

['product_id', 'productName', 'stock', 'last_stocked', 'warehouse', 'cost_price', 'available']


SynapseWidget(Synapse.DataFrame, 151506fb-f568-48db-b73d-06088f67f105)

In [40]:
df_inventory = (
    df_inventory_raw
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("last_stocked", "LastStocked")
    .withColumn(
        "Stock",
        when(col("stock").cast("int").isNotNull(),
             col("stock").cast("int"))
        .when(col("stock") == "twenty five", lit(25))
        .when(col("stock") == "twenty", lit(20))
        .when(col("stock") == "eighteen", lit(18))
        .when(col("stock") == "fifteen", lit(15))
        .when(col("stock") == "twelve", lit(12))
        .otherwise(lit(0))
    )
    .withColumn(
        "CostPrice",
        regexp_extract(
            col("CostPrice").cast("string"),
            r"([0-9]+\.?[0-9]*)",
            1
        ).cast("double")
    )
    .drop("stock")
)

# Remove the incorrect Silver table
spark.sql("DROP TABLE IF EXISTS silver_inventory")

# Save the corrected table
df_inventory.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_inventory")

# Verify
display(spark.table("silver_inventory"))

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 239e023f-eeb9-461b-ba96-77c86605401e)

In [41]:
from pyspark.sql.functions import col

# Read the ORIGINAL Bronze inventory file
df_inventory_raw = spark.read.parquet(
    "Files/Bronze/inventory_data.parquet"
)

# Check the columns
print(df_inventory_raw.columns)

# Show the data
display(df_inventory_raw)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 43, Finished, Available, Finished, False)

['product_id', 'productName', 'stock', 'last_stocked', 'warehouse', 'cost_price', 'available']


SynapseWidget(Synapse.DataFrame, 0e782198-119c-403d-8f4a-ba1273ad9667)

In [42]:
from pyspark.sql.functions import col, when, lit, regexp_extract

# Clean the original Bronze inventory data
df_inventory = (
    df_inventory_raw
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("last_stocked", "LastStocked")
    
    # Convert Stock values into numbers
    .withColumn(
        "Stock",
        when(col("stock").cast("int").isNotNull(),
             col("stock").cast("int"))
        .when(col("stock") == "twenty five", lit(25))
        .when(col("stock") == "twenty", lit(20))
        .when(col("stock") == "eighteen", lit(18))
        .when(col("stock") == "fifteen", lit(15))
        .when(col("stock") == "twelve", lit(12))
        .otherwise(lit(0))
    )
    
    # Convert CostPrice to number
    .withColumn(
        "CostPrice",
        regexp_extract(
            col("CostPrice").cast("string"),
            r"([0-9]+\.?[0-9]*)",
            1
        ).cast("double")
    )
    
    # Remove the old stock column
    .drop("stock")
)

# Delete the incorrect Silver table
spark.sql("DROP TABLE IF EXISTS silver_inventory")

# Save the corrected Silver Inventory table
df_inventory.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_inventory")

# Verify the result
display(spark.table("silver_inventory"))

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 89324f9c-d30a-4e45-84d3-e5c78f1ee430)

In [45]:
from pyspark.sql.functions import col, when, lit, lower, trim, regexp_extract

# Read the Bronze file FRESH — don't use df_inventory_raw
inventory_source = spark.read.parquet(
    "Files/Bronze/inventory_data.parquet"
)

# Show the actual Bronze columns
print("BRONZE COLUMNS:")
print(inventory_source.columns)

# Create Stock explicitly
df_inventory_new = (
    inventory_source
    .select(
        col("product_id"),
        col("productName").alias("ProductName"),
        col("last_stocked").alias("LastStocked"),
        col("warehouse"),
        col("cost_price").alias("CostPrice"),
        col("stock"),
        col("available")
    )
    .withColumn(
        "Stock",
        when(
            trim(col("stock").cast("string")).rlike("^[0-9]+$"),
            col("stock").cast("int")
        )
        .when(lower(trim(col("stock").cast("string"))) == "twenty five", lit(25))
        .when(lower(trim(col("stock").cast("string"))) == "twenty", lit(20))
        .when(lower(trim(col("stock").cast("string"))) == "eighteen", lit(18))
        .when(lower(trim(col("stock").cast("string"))) == "fifteen", lit(15))
        .when(lower(trim(col("stock").cast("string"))) == "twelve", lit(12))
        .otherwise(lit(0))
    )
    .withColumn(
        "CostPrice",
        regexp_extract(
            col("CostPrice").cast("string"),
            r"([0-9]+\.?[0-9]*)",
            1
        ).cast("double")
    )
    .drop("stock")
)

print("NEW INVENTORY COLUMNS:")
print(df_inventory_new.columns)

display(df_inventory_new)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 47, Finished, Available, Finished, False)

BRONZE COLUMNS:
['product_id', 'productName', 'stock', 'last_stocked', 'warehouse', 'cost_price', 'available']
NEW INVENTORY COLUMNS:
['product_id', 'ProductName', 'LastStocked', 'warehouse', 'CostPrice', 'available']


SynapseWidget(Synapse.DataFrame, 5f0b6b7a-546f-4034-b384-f4e05970947d)

In [46]:
# Remove the old incorrect Silver Inventory table
spark.sql("DROP TABLE IF EXISTS silver_inventory")

# Save the corrected inventory table
df_inventory_new.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_inventory")

# Verify the saved table
df_inventory_saved = spark.table("silver_inventory")

print("Silver Inventory columns:")
print(df_inventory_saved.columns)

display(df_inventory_saved)


StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 48, Finished, Available, Finished, False)

Silver Inventory columns:
['product_id', 'ProductName', 'LastStocked', 'warehouse', 'CostPrice', 'available']


SynapseWidget(Synapse.DataFrame, 098d8435-efc8-4ad0-8488-4c2ead0f9381)

In [47]:
from pyspark.sql.functions import (
    col, sum, avg, countDistinct, when,
    coalesce, lit, round, max
)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 49, Finished, Available, Finished, False)

In [50]:
from pyspark.sql.functions import (
    col, sum, avg, countDistinct, when,
    coalesce, lit, round, max
)

# Load Silver tables
df_orders = spark.table("silver_orders")
df_inventory = spark.table("silver_inventory")
df_returns = spark.table("silver_returns")

# Fix Order ID names
if "Order_ID" in df_orders.columns:
    df_orders = df_orders.withColumnRenamed("Order_ID", "OrderID")

if "Order_ID" in df_returns.columns:
    df_returns = df_returns.withColumnRenamed("Order_ID", "OrderID")

# -----------------------------
# RETURNS SUMMARY
# -----------------------------

df_return_summary = (
    df_returns
    .groupBy("OrderID")
    .agg(
        countDistinct("ReturnID").alias("Return_Count"),
        round(sum("ReturnAmount"), 2).alias("Total_Return_Amount")
    )
)

# -----------------------------
# ORDERS + RETURNS
# -----------------------------

df_gold = (
    df_orders
    .join(
        df_return_summary,
        on="OrderID",
        how="left"
    )
    .withColumn(
        "Return_Count",
        coalesce(col("Return_Count"), lit(0))
    )
    .withColumn(
        "Total_Return_Amount",
        coalesce(
            col("Total_Return_Amount"),
            lit(0.0)
        )
    )
)

# -----------------------------
# JOIN INVENTORY
# -----------------------------

df_gold = df_gold.join(
    df_inventory.select(
        "ProductName",
        "CostPrice"
    ),
    on="ProductName",
    how="left"
)

# -----------------------------
# COGS
# -----------------------------

df_gold = df_gold.withColumn(
    "COGS",
    round(
        col("Qty") * col("CostPrice"),
        2
    )
)

# -----------------------------
# NET PROFIT
# -----------------------------

df_gold = df_gold.withColumn(
    "Net_Profit",
    round(
        col("OrderAmount") - col("COGS"),
        2
    )
)

# -----------------------------
# PRODUCT KPIs
# -----------------------------

df_kpi = (
    df_gold
    .groupBy("ProductName")
    .agg(
        countDistinct("OrderID").alias("Total_Orders"),
        countDistinct("CustomerID").alias("Unique_Customers"),
        sum("Qty").alias("Total_Quantity_Sold"),
        round(sum("OrderAmount"), 2).alias("Total_Revenue"),
        round(avg("OrderAmount"), 2).alias("Avg_Order_Value"),
        round(sum("COGS"), 2).alias("Total_COGS"),
        round(avg("CostPrice"), 2).alias("Avg_Cost"),
        round(sum("Net_Profit"), 2).alias("Net_Profit"),
        sum("Return_Count").alias("Total_Returns"),
        countDistinct(
            when(
                col("Return_Count") > 0,
                col("OrderID")
            )
        ).alias("Returned_Orders"),
        round(
            sum("Total_Return_Amount"),
            2
        ).alias("Total_Return_Amount")
    )
)

# -----------------------------
# RETURN RATE
# -----------------------------

df_kpi = df_kpi.withColumn(
    "Return_Rate_Percent",
    round(
        col("Returned_Orders") /
        col("Total_Orders") * 100,
        2
    )
)

# -----------------------------
# SAVE GOLD TABLE
# -----------------------------

df_kpi.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_product_kpis")

# -----------------------------
# DISPLAY RESULT
# -----------------------------

display(df_kpi)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 85328f1c-27bd-4e9e-afcd-28cefb8a7b37)

In [51]:
display(df_kpi)

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad1a6ba1-301f-4fb1-a5c5-50afc84e41ba)

In [52]:
# Save Silver tables into Files/Silver

spark.table("silver_orders") \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .save("Files/Silver/orders")

spark.table("silver_inventory") \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .save("Files/Silver/inventory")

spark.table("silver_returns") \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .save("Files/Silver/returns")


# Save Gold table into Files/Gold

spark.table("gold_product_kpis") \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .save("Files/Gold/product_kpis")

print("Silver and Gold files saved successfully!")

StatementMeta(, 292975af-3329-446d-9cae-20e55f16a4aa, 54, Finished, Available, Finished, False)

Silver and Gold files saved successfully!
